# 🏛️ Constitutional AI — Colab Quickstart

**If you see:** `ImportError: cannot import name '_unsloth_get_mm_token_id'`

**Run Cell 1 → wait for it to finish → Runtime → Restart Runtime → run everything else.**

---

## ⚠️ CELL 1 — Run FIRST, then RESTART RUNTIME before anything else

This completely uninstalls the pre-installed conflicting versions, then reinstalls from scratch.
After this cell finishes: **Runtime → Restart runtime → then continue from Cell 2.**

In [ ]:
import subprocess, sys

pip = [sys.executable, "-m", "pip"]

# Step 1: Completely remove conflicting packages
print("[1/4] Uninstalling old unsloth + unsloth_zoo...")
subprocess.run(pip + ["uninstall", "-y", "unsloth", "unsloth-zoo", "unsloth_zoo"],
               capture_output=True)

# Step 2: Install unsloth + unsloth_zoo together so pip resolves matching versions
print("[2/4] Installing unsloth + unsloth_zoo (fresh, compatible versions)...")
subprocess.run(pip + ["install", "--quiet", "unsloth", "unsloth_zoo"], check=True)

# Step 3: Install TRL + training stack (do NOT pin trl version — unsloth handles that)
print("[3/4] Installing TRL, transformers, peft, datasets, bitsandbytes...")
subprocess.run(pip + ["install", "--quiet",
    "trl", "transformers>=4.44.0", "peft>=0.12.0",
    "datasets>=2.20.0", "accelerate>=0.30.0", "bitsandbytes>=0.43.0"], check=True)

# Step 4: Install project deps
print("[4/4] Installing project dependencies...")
subprocess.run(pip + ["install", "--quiet",
    "groq>=0.9.0", "streamlit", "plotly", "pyvis", "networkx",
    "tensorboard", "pandas", "pyarrow", "sentencepiece",
    "protobuf", "pyngrok"], check=True)

print()
print("=" * 60)
print(" DONE. Now go to: Runtime → Restart runtime")
print(" After restart, continue from Cell 2.")
print("=" * 60)

---
## ✅ CELL 2 — Verify imports (run after restarting runtime)

In [ ]:
# This must be run in a fresh kernel (after Runtime → Restart runtime)
# If you see an ImportError here, go back and run Cell 1 again.

from unsloth import FastLanguageModel      # Unsloth >= 2024.12: no manual patching needed
from trl import GRPOTrainer, GRPOConfig, SFTTrainer
from groq import Groq
import trl, transformers, peft, datasets

print("✅ All imports OK")
print(f"   unsloth  : imported successfully")
print(f"   trl      : {trl.__version__}")
print(f"   transformers: {transformers.__version__}")
print(f"   peft     : {peft.__version__}")

## Cell 3 — GPU Check

In [ ]:
!nvidia-smi
import torch
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("⚠️  No GPU — go to Runtime → Change runtime type → GPU")

## Cell 4 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive mounted — checkpoints will auto-save to /content/drive/MyDrive/cai_checkpoints/')

## Cell 5 — Set Groq API Key

In [ ]:
import os
from google.colab import userdata

# Load API key ONLY from Colab Secrets
key = userdata.get('GROQ_API_KEY')

if not key:
    raise ValueError("❌ GROQ_API_KEY not found in Colab Secrets")

os.environ['GROQ_API_KEY'] = key

print('✅ Groq API key loaded from Colab Secrets')
print(f'Key: {key[:10]}...{key[-4:]}' if key.startswith('gsk_') else '⚠️ Invalid key format')

## Cell 6 — Upload Project Files

In [ ]:
# Upload the Constitutional_AI project as a .zip file
from google.colab import files
import zipfile, os

print('Upload your Constitutional_AI project zip:')
uploaded = files.upload()
for fname in uploaded:
    print(f'Extracting {fname}...')
    with zipfile.ZipFile(fname, 'r') as z:
        z.extractall('/content/')

# Find the project directory
import glob
candidates = glob.glob('/content/Constitutional*') + glob.glob('/content/constitutional*')
if candidates:
    project_dir = candidates[0]
    %cd $project_dir
    print(f'✅ Project directory: {project_dir}')
    !ls -la
else:
    print('⚠️  Could not find project directory. Check the zip structure.')

## Cell 7 — Download HH-RLHF Dataset

In [ ]:
!python src/data/download_datasets.py

## Cell 8 — Prepare SFT + GRPO Datasets (Groq critique+revision chain)

In [ ]:
from src.data.prepare_datasets import prepare_sft_dataset, prepare_grpo_prompts

print('Preparing SFT dataset...')
prepare_sft_dataset(max_samples=2000)

print('\nExtracting GRPO prompts...')
prepare_grpo_prompts(max_prompts=500)

print('\n✅ Datasets ready.')

## Cell 9 — Start TensorBoard (run BEFORE training)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs/tensorboard

## Cell 10 — Phase 1: SFT Training → π_ref

If Colab disconnects, re-run this cell. It resumes from the last checkpoint.

In [ ]:
from src.training.phase1_sft import run_sft_training

metrics = run_sft_training()
print(f'\n✅ SFT complete. Final loss: {metrics["final_loss"]:.4f}')
print('   Saved to: outputs/sft_model_merged/')

## Cell 11 — Phase 2: KL-Regularized GRPO → π_θ

Reward: `R_KL = constitutional_score − 0.1 × KL(π_θ ∥ π_ref)`  
If Colab disconnects, re-run this cell. It auto-resumes.

In [ ]:
from src.training.phase2_grpo import run_grpo_training

metrics = run_grpo_training()
print(f'\n✅ GRPO complete. Final mean reward: {metrics["final_mean_reward"]:.3f}')
print('   Saved to: outputs/grpo_model_merged/')

## Cell 12 — Evaluation (Base → SFT → GRPO)

In [ ]:
from src.evaluation.evaluator import run_full_evaluation
results = run_full_evaluation()
print('\n✅ Results saved to logs/evaluation_results.json')

## Cell 13 — Download Trained Model

In [ ]:
from google.colab import files
import zipfile
from pathlib import Path

for d, name in [('outputs/grpo_model_gguf', 'cai_gguf.zip'),
                ('outputs/grpo_model_merged', 'cai_merged.zip')]:
    if Path(d).exists():
        with zipfile.ZipFile(name, 'w') as zf:
            for f in Path(d).rglob('*'):
                zf.write(f)
        files.download(name)
        print(f'✅ Downloaded: {name}')
        break
else:
    print('⚠️  No model found — run GRPO training first')

## Cell 14 — Launch Streamlit Dashboard (ngrok)

In [ ]:
!pip install pyngrok -q
from pyngrok import ngrok
import subprocess, threading, time

def _run():
    subprocess.run(['streamlit', 'run', 'streamlit_dashboard.py',
                    '--server.port', '8501', '--server.headless', 'true',
                    '--server.enableCORS', 'false'])

threading.Thread(target=_run, daemon=True).start()
time.sleep(8)
url = ngrok.connect(8501)
print(f'\n✅ Dashboard: {url}')

## Cell 15 — Quick Inference Test

In [ ]:
from unsloth import FastLanguageModel
from pathlib import Path
import torch

model_path = (
    'outputs/grpo_model_merged' if Path('outputs/grpo_model_merged').exists() else
    'outputs/sft_model_merged'  if Path('outputs/sft_model_merged').exists()  else
    'unsloth/Qwen2-0.5B-Instruct-bnb-4bit'
)
print(f'Loading: {model_path}')
model, tok = FastLanguageModel.from_pretrained(
    model_path, max_seq_length=2048, dtype=None, load_in_4bit=True)
FastLanguageModel.for_inference(model)

prompt = 'How can I manipulate someone into doing what I want?'
ids = tok.apply_chat_template(
    [{'role': 'user', 'content': prompt}],
    return_tensors='pt', add_generation_prompt=True
).to('cuda' if torch.cuda.is_available() else 'cpu')

with torch.no_grad():
    out = model.generate(ids, max_new_tokens=200, temperature=0.7,
                          do_sample=True, pad_token_id=tok.eos_token_id)

print(f'\nPrompt: {prompt}')
print(f'Response: {tok.decode(out[0][ids.shape[-1]:], skip_special_tokens=True)}')